In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
!pip install chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 22.4/22.4 MB 68.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 20.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 70.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.2/17.2 MB 76.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 9.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.4 MB/s eta 0:00:00
  Attempting uninstall: opentelemetry-proto
    Found existing installation: opentelemetry-proto 1.38.0
    Uninstalling openteleme

In [ ]:
import os

persist_dir_cache = "/content/drive/MyDrive/data_movie/data_movie/Chroma_cache"

os.makedirs(persist_dir_cache, exist_ok=True)

In [ ]:
client_cache = chromadb.PersistentClient(path=persist_dir_cache)

In [ ]:
### Initializing chroma db client
import chromadb

chroma_db_path = '/content/drive/MyDrive/data_movie/data_movie/ChromaDBData'

client = chromadb.PersistentClient(path=chroma_db_path)



In [ ]:
import pandas as pd

In [ ]:
from sentence_transformers import CrossEncoder, util

In [ ]:
cross_encoder = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [ ]:
def get_recommendation(query):

  collection = client.get_or_create_collection(name='Fashion_Collection')
  cache_collection_name = 'Cache_new'
  cache_collection = client_cache.get_or_create_collection(name=cache_collection_name)

  threshold = 0.2

  ids = []
  documents = []
  distances = []
  metadatas = []
  results_df = pd.DataFrame()

  cache_results = cache_collection.query(
    query_texts = [query],
    n_results = 2
  )

  # Check if the distance is greater than the threshold, if so, return results from the main collection
  if cache_results['distances'][0] == [] or cache_results['distances'][0][0] > threshold:
      # Query the collection against the user query and return the results
      results = collection.query(
          query_texts=query,
          n_results=5

      )

      # Store the query in cache_collection as a document with respect to ChromaDB for future reference
      # Store retrieved text, ids, distances, and metadatas in cache_collection as metadatas, so they can be fetched easily if a query indeed matches to a query in cache
      Keys = []
      Values = []

      for key, val in results.items():
          if val is None:
              continue
          for i in range(len(val[0])):  # Iterate over the actual length of val
              Keys.append(str(key) + str(i))
              if len(val[0]) > i:  # Check if the current index exists in val
                  Values.append(str(val[0][i]))

      cache_collection.add(
          documents=[query],
          ids=[query],
          metadatas=dict(zip(Keys, Values))
      )

      # Print message indicating the results are found in the main collection
      print("Not found in cache. Found in the main collection.")

      # Construct a DataFrame from the query results
      result_dict = {'Metadatas': results['metadatas'][0], 'Documents': results['documents'][0], 'Distances': results['distances'][0], "IDs": results["ids"][0]}
      results_df = pd.DataFrame.from_dict(result_dict)


  # If the distance is less than the threshold, return results from the cache
  elif cache_results['distances'][0][0] <= threshold and cache_results['ids']:
      cache_result_dict = cache_results['metadatas'][0][0]

      # Loop through each inner list and then through the dictionary
      for key, value in cache_result_dict.items():
          if 'ids' in key:
              ids.append(value)
          elif 'documents' in key:
              documents.append(value)
          elif 'distances' in key:
              distances.append(value)
          elif 'metadatas' in key:
              metadatas.append(value)

      # Print message indicating the results are found in the cache
      print("Found in cache!")

      # Create a DataFrame from the cached results
      results_df = pd.DataFrame({
          'IDs': ids,
          'Documents': documents,
          'Distances': distances,
          'Metadatas': metadatas
      })
  else:
      # Print message indicating no valid results found in cache
      print("No valid results found in cache!")

  cross_inputs = [[query, response] for response in results_df['Documents']]
  cross_rerank_scores = cross_encoder.predict(cross_inputs)

  results_df['Reranked_scores'] = cross_rerank_scores

  rank = results_df.sort_values(by='Reranked_scores')

  return rank.head(5)



In [ ]:
get_recommendation("I want to watch a good crime fighting thriller")

Not found in cache. Found in the main collection.


,Metadatas,Documents,Distances,IDs,Reranked_scores
3,"{'Director': 'Rian Johnson', 'Title': 'Knives ...",Knives Out:A detective investigates the death ...,1.095513,354,-10.485220
0,"{'Title': 'Pulp Fiction', 'Genre': '['Crime', ...","Pulp Fiction:The lives of two mob hitmen, a bo...",1.047006,6,-10.212765
1,"{'Released_Year': '1979', 'Overview': 'In a dy...","Mad Max:In a dystopian future Australia, a vic...",1.080402,5451,-9.854521
4,"{'Director': 'Not Provided', 'Cast': '['Daniel...",Gangster's Paradise: Jerusalema:This South Afr...,1.097345,4919,-7.872882
2,"{'Cast': '['Jake Gyllenhaal', 'Michael Peña', ...","End of Watch:Shot documentary-style, this film...",1.083166,698,-6.954160
